In [2]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from astropy.table import Table
from astropy.table import vstack as tab_vstack
from astropy.table import join as tab_join

from lvmdap.dap_tools import list_columns,read_DAP_file,map_plot_DAP,scatter
from matplotlib import use as mpl_use
#mpl_use('Agg')
from lvmdap.dap_tools import plot_spectra, read_coeffs_RSP, read_elines_RSP, read_tab_EL

%matplotlib inline
from matplotlib import rcParams as rc
rc.update({'font.size': 19,\
           'font.weight': 900,\
           'text.usetex': True,\
           'path.simplify'           :   True,\
           'xtick.labelsize' : 19,\
           'ytick.labelsize' : 19,\
#           'xtick.major.size' : 3.5,\
#           'ytick.major.size' : 3.5,\
           'axes.linewidth'  : 2.0,\
               # Increase the tick-mark lengths (defaults are 4 and 2)
           'xtick.major.size'        :   6,\
           'ytick.major.size'        :   6,\
           'xtick.minor.size'        :   3,\
           'ytick.minor.size'        :   3,\
           'xtick.major.width'       :   1,\
           'ytick.major.width'       :   1,\
           'lines.markeredgewidth'   :   1,\
           'legend.numpoints'        :   1,\
           'xtick.minor.width'       :   1,\
           'ytick.minor.width'       :   1,\
           'legend.frameon'          :   False,\
           'legend.handletextpad'    :   0.3,\
           'font.family'    :   'serif',\
           'mathtext.fontset'        :   'stix',\
           'axes.facecolor' : "w",\
           
          })
import math
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.colors as mpl_colors
import datetime as dt
from astropy.table import MaskedColumn
DRPVER = "1.2.0"
DAPVER = '1.2.0.251218' 

In [3]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, Iterable, Optional, Union

import pandas as pd
import yaml

In [4]:
def _as_list(x):
    if x is None:
        return []
    if isinstance(x, (list, tuple, set)):
        return list(x)
    return [x]


def _safe_get(dct: dict, *keys, default=None):
    cur = dct
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur


def _validate_basic_datamodel_structure(data: dict) -> list[str]:
    """
    Basic structural validation for the SDSS-style YAML datamodel.
    This is not a full schema validator, but it catches common problems.
    """
    errors = []

    if not isinstance(data, dict):
        return ["Top-level object is not a mapping/dictionary."]

    required_top = ["general", "releases"]
    for key in required_top:
        if key not in data:
            errors.append(f"Missing top-level key: {key}")

    general = data.get("general")
    if general is not None and not isinstance(general, dict):
        errors.append("Top-level key 'general' must be a mapping.")
    elif isinstance(general, dict):
        for key in ["name", "datatype"]:
            if key not in general:
                errors.append(f"Missing general.{key}")

    releases = data.get("releases")
    if releases is not None and not isinstance(releases, dict):
        errors.append("Top-level key 'releases' must be a mapping.")
    elif isinstance(releases, dict):
        if len(releases) == 0:
            errors.append("No releases found under 'releases'.")
        for rel_name, rel_data in releases.items():
            if not isinstance(rel_data, dict):
                errors.append(f"Release '{rel_name}' must be a mapping.")
                continue
            for key in ["template", "location", "hdus"]:
                if key not in rel_data:
                    errors.append(f"Missing releases.{rel_name}.{key}")
            hdus = rel_data.get("hdus")
            if hdus is not None and not isinstance(hdus, dict):
                errors.append(f"releases.{rel_name}.hdus must be a mapping.")
            elif isinstance(hdus, dict):
                for hdu_name, hdu_data in hdus.items():
                    if not isinstance(hdu_data, dict):
                        errors.append(
                            f"releases.{rel_name}.hdus.{hdu_name} must be a mapping."
                        )
                        continue
                    for key in ["name", "description", "is_image"]:
                        if key not in hdu_data:
                            errors.append(
                                f"Missing releases.{rel_name}.hdus.{hdu_name}.{key}"
                            )

    return errors


def _general_table(data: dict) -> pd.DataFrame:
    general = data.get("general", {})
    rows = [{"field": k, "value": v} for k, v in general.items()]
    return pd.DataFrame(rows)


def _release_table(data: dict) -> pd.DataFrame:
    rows = []
    for rel_name, rel_data in data.get("releases", {}).items():
        rows.append(
            {
                "release": rel_name,
                "template": rel_data.get("template"),
                "example": rel_data.get("example"),
                "location": rel_data.get("location"),
                "environment": rel_data.get("environment"),
                "survey": rel_data.get("survey"),
                "n_hdus": len(rel_data.get("hdus", {}))
                if isinstance(rel_data.get("hdus", {}), dict)
                else None,
            }
        )
    return pd.DataFrame(rows)


def _hdu_summary_table(data: dict, release: str) -> pd.DataFrame:
    hdus = _safe_get(data, "releases", release, "hdus", default={}) or {}
    rows = []
    for hdu_key, hdu_data in hdus.items():
        rows.append(
            {
                "hdu_key": hdu_key,
                "name": hdu_data.get("name"),
                "description": hdu_data.get("description"),
                "is_image": hdu_data.get("is_image"),
                "size": hdu_data.get("size"),
                "n_header_cards": len(hdu_data.get("header", []))
                if isinstance(hdu_data.get("header", []), list)
                else None,
                "n_columns": len(hdu_data.get("columns", {}))
                if isinstance(hdu_data.get("columns", {}), dict)
                else None,
            }
        )
    return pd.DataFrame(rows)


def _header_table(data: dict, release: str, hdu_key: str) -> pd.DataFrame:
    header = _safe_get(data, "releases", release, "hdus", hdu_key, "header", default=[])
    rows = []
    for item in header:
        if isinstance(item, dict):
            rows.append(
                {
                    "key": item.get("key"),
                    "value": item.get("value"),
                    "comment": item.get("comment"),
                }
            )
    return pd.DataFrame(rows)


def _columns_table(data: dict, release: str, hdu_key: str) -> pd.DataFrame:
    columns = _safe_get(
        data, "releases", release, "hdus", hdu_key, "columns", default={}
    ) or {}
    rows = []
    for col_key, col_data in columns.items():
        if isinstance(col_data, dict):
            rows.append(
                {
                    "column_key": col_key,
                    "name": col_data.get("name"),
                    "type": col_data.get("type"),
                    "unit": col_data.get("unit"),
                    "description": col_data.get("description"),
                }
            )
    return pd.DataFrame(rows)


def inspect_datamodel_yaml(
    paths: Union[str, Path, Iterable[Union[str, Path]]],
    release: Optional[str] = None,
    show: bool = True,
    max_rows: Optional[int] = None,
) -> Dict[str, Dict[str, Any]]:
    """
    Read one or more SDSS-style YAML datamodel files, validate basic structure,
    and return their contents as tables.

    Parameters
    ----------
    paths
        A single path or a list of paths.
    release
        Specific release to inspect. If None, the first release found is used.
    show
        If True, print tables to screen.
    max_rows
        Optional truncation for displayed rows.

    Returns
    -------
    results : dict
        Nested dictionary with parsed YAML, validation info, and pandas tables.
    """
    results: Dict[str, Dict[str, Any]] = {}

    for path_like in _as_list(paths):
        path = Path(path_like)
        result: Dict[str, Any] = {
            "path": str(path),
            "exists": path.exists(),
            "valid_yaml": False,
            "valid_structure": False,
            "yaml_error": None,
            "structure_errors": [],
            "data": None,
            "tables": {},
        }

        if not path.exists():
            result["yaml_error"] = "File does not exist."
            results[str(path)] = result
            continue

        try:
            with open(path, "r", encoding="utf-8") as f:
                data = yaml.safe_load(f)
            result["valid_yaml"] = True
            result["data"] = data
        except yaml.YAMLError as e:
            result["yaml_error"] = f"YAML parsing error: {e}"
            results[str(path)] = result
            continue
        except Exception as e:
            result["yaml_error"] = f"Unexpected read error: {e}"
            results[str(path)] = result
            continue

        structure_errors = _validate_basic_datamodel_structure(data)
        result["structure_errors"] = structure_errors
        result["valid_structure"] = len(structure_errors) == 0

        # Build tables
        result["tables"]["general"] = _general_table(data)
        result["tables"]["releases"] = _release_table(data)

        releases = list((data.get("releases") or {}).keys())
        chosen_release = release or (releases[0] if releases else None)
        result["chosen_release"] = chosen_release

        if chosen_release is not None:
            result["tables"]["hdu_summary"] = _hdu_summary_table(data, chosen_release)

            hdu_tables = {}
            hdus = _safe_get(data, "releases", chosen_release, "hdus", default={}) or {}
            for hdu_key in hdus:
                hdu_tables[hdu_key] = {
                    "header": _header_table(data, chosen_release, hdu_key),
                    "columns": _columns_table(data, chosen_release, hdu_key),
                }
            result["tables"]["hdus"] = hdu_tables

        results[str(path)] = result

        if show:
            print("=" * 100)
            print(f"FILE: {path}")
            print(f"VALID YAML: {result['valid_yaml']}")
            print(f"VALID STRUCTURE: {result['valid_structure']}")
            if result["yaml_error"]:
                print(f"ERROR: {result['yaml_error']}")
                continue
            if result["structure_errors"]:
                print("STRUCTURE ERRORS:")
                for err in result["structure_errors"]:
                    print(f"  - {err}")

            def _show_df(title: str, df: pd.DataFrame):
                print(f"\n--- {title} ---")
                if df is None or df.empty:
                    print("[empty]")
                else:
                    if max_rows is not None:
                        print(df.head(max_rows).to_string(index=False))
                    else:
                        print(df.to_string(index=False))

            _show_df("GENERAL", result["tables"]["general"])
            _show_df("RELEASES", result["tables"]["releases"])

            if chosen_release is not None:
                _show_df(f"HDU SUMMARY ({chosen_release})", result["tables"]["hdu_summary"])
                for hdu_key, tables in result["tables"]["hdus"].items():
                    _show_df(f"{hdu_key} HEADER", tables["header"])
                    _show_df(f"{hdu_key} COLUMNS", tables["columns"])

    return results

In [8]:
yaml_file='/home/sanchez/sda2/code/python/datamodel/datamodel/products/yaml/lvm_dapall.yaml'  
#yaml_file='/home/sanchez/sda2/code/python/datamodel/datamodel/products/yaml/lvm_lv_dap.yaml'
#yaml_file='/home/sanchez/sda2/code/python/datamodel/datamodel/products/yaml/lvm_dap.yaml'     
#yaml_file='/home/sanchez/sda2/code/python/datamodel/datamodel/products/yaml/lvm_sframe.yaml'
#yaml_file='/home/sanchez/sda2/code/python/datamodel/datamodel/products/yaml/lvm_drpall.yaml'

#yaml_file = '/home/sanchez/sda2/code/python/datamodel/datamodel/products/yaml/lvm_drpall.yaml'
inspect_datamodel_yaml(
        yaml_file,
        show=True,
        max_rows=20   # optional
    )

FILE: /home/sanchez/sda2/code/python/datamodel/datamodel/products/yaml/lvm_dapall.yaml
VALID YAML: True
VALID STRUCTURE: True

--- GENERAL ---
                      field                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

{'/home/sanchez/sda2/code/python/datamodel/datamodel/products/yaml/lvm_dapall.yaml': {'path': '/home/sanchez/sda2/code/python/datamodel/datamodel/products/yaml/lvm_dapall.yaml',
  'exists': True,
  'valid_yaml': True,
  'valid_structure': True,
  'yaml_error': None,
  'structure_errors': [],
  'data': {'general': {'name': 'lvm_dapall',
    'short': 'Summary table of LVM DR20 DAP products and exposure-averaged spectral properties.',
    'description': 'The file `dapall-{drpver}-{dapver}.fits` is a summary of the dataproducts produced by the LVM DAP analysis for the delivered exposures included in SDSS-V DR20. It comprises a binary table in which each row corresponds to an individual exposure identified by `tileid`, `mjd`, and `expnum`, together with the corresponding DAP file, and contains observational and physical properties derived from stellar-continuum fitting and emission-line measurements of the observed spectra. Fluxes listed in this file are average values within the field of v